# P vs NP – easy to check, hard to find

* **P** – problems we can *solve* in polynomial time ($n$, $n^2$, $n^3$, ...): sorting, shortest path, ...
* **NP** – problems where a proposed solution (a *certificate*) can be *verified* in polynomial time.
  Finding it may take exponential time ($2^n$, $n!$) – as far as anybody knows.
* **NP-complete** – the hardest problems in NP. Every NP problem can be translated (*reduced*) to any of them,
  so a fast algorithm for one of them would give a fast algorithm for all of them.

Whether $P = NP$ is the most famous open question in computer science (Millennium Prize, \$1 000 000).
Almost everybody believes $P \neq NP$ – but nobody has proven it.

The trap for a programmer: NP-complete problems often have **one-line assignments** that look trivial.
In this notebook we look at several of them – always with the same pattern:

> *checking* a solution is a few lines of fast code – *finding* it takes longer than the age of the Universe.

*How to read the code:* read the comments; the exact Python syntax is secondary.

### The precise definition (worth getting right)

A problem is **NP-complete** when it satisfies *both* of these conditions:

1. it is **in NP** – a proposed solution can be verified in polynomial time, and
2. it is **NP-hard** – every problem in NP can be translated (*reduced*) to it in polynomial time.

Condition 2 alone gives an **NP-hard** problem; such a problem does not have to be in NP at all.
Only *decision* problems (answer yes / no) can be NP-complete, because only they have a certificate to verify.

**TSP as an example:**

| question | class | why |
|---|---|---|
| "Is there a round trip shorter than 500 km?" | **NP-complete** | in NP: the certificate is the trip, we add up its legs; NP-hard: the Hamiltonian cycle problem reduces to it |
| "Find the shortest round trip." | **NP-hard**, not known to be in NP | not a yes/no question – and there is no known short certificate that a trip is *the shortest*: you would have to rule out every shorter trip |

In practice the two versions are equally hard (asking "shorter than L?" for a few values of L finds the optimum),
so people say "TSP is NP-complete" as a shorthand. Strictly: the decision version is NP-complete, the optimisation version is NP-hard.
An extreme example of an NP-hard problem outside NP is the halting problem, which cannot be solved at all.

In [ ]:
import itertools                 # combinations, permutations, products
import math
import random
from time import perf_counter    # stopwatch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx            # graphs

YEAR = 365.25 * 24 * 3600
AGE_OF_UNIVERSE = 13.8e9 * YEAR      # seconds since the Big Bang


def human_time(seconds: float) -> str:
    """Convert seconds into readable text: '3.2 ms', '1.7 days', '6.4 thousand years', ..."""
    units = [("ns", 1e-9), ("µs", 1e-6), ("ms", 1e-3), ("s", 1), ("min", 60), ("h", 3600),
             ("days", 86400), ("years", YEAR), ("thousand years", 1e3 * YEAR),
             ("million years", 1e6 * YEAR), ("billion years", 1e9 * YEAR)]
    if seconds > 1e6 * 1e9 * YEAR:
        return f"{seconds / (1e9 * YEAR):.1e} billion years"
    for name, size in reversed(units):
        if seconds >= size:
            return f"{seconds / size:,.1f} {name}"
    return f"{seconds / 1e-9:.2f} ns"

## 0. Where does the TSP experience fit? Growth classes at a glance

In the TSP notebook we met $(n-1)!$ and $n^2 \cdot 2^n$. Let us put them next to the other common growth rates.
The table assumes **1 microsecond per elementary operation** (roughly one Python bytecode step; C is ~100× faster,
which – as we saw – shifts the picture by only a few items).

| class | typical example from this course |
|---|---|
| $\log n$ | binary search in a sorted list |
| $n$ | Euler circuit test (count degrees), verifying a certificate |
| $n \log n$ | sorting, Dijkstra / A* with a heap (per edge) |
| $n^2$ | nearest-neighbour TSP heuristic, 2-opt pass, Prim on a dense graph |
| $n^3$ | Floyd–Warshall all-pairs shortest paths, matrix multiplication (naive) |
| $2^n$ | subset sum by brute force, 3-colouring is $3^n$ |
| $n^2 \cdot 2^n$ | Held–Karp exact TSP |
| $n!$ | brute-force TSP, brute-force Hamiltonian cycle |

Everything above the line $2^n$ is **polynomial** – doubling the input multiplies the work by a constant (2, 4, 8).
Everything from $2^n$ down is **exponential** or worse – *adding one item* multiplies the work by a constant or more.

In [ ]:
def log10_time_to_text(log10_seconds: float) -> str:
    """Readable time from log10(seconds). We work with logarithms because 1000! is far too big for a normal number."""
    if log10_seconds < 25:
        return human_time(10 ** log10_seconds)
    return f"10^{log10_seconds - math.log10(YEAR):.0f} years"


MICROSECOND = 1e-6
# For every growth class: a function that returns log10(number of operations) for input size n
growth = {
    "log n":     lambda n: math.log10(math.log2(n)),
    "n":         lambda n: math.log10(n),
    "n log n":   lambda n: math.log10(n * math.log2(n)),
    "n^2":       lambda n: 2 * math.log10(n),
    "n^3":       lambda n: 3 * math.log10(n),
    "2^n":       lambda n: n * math.log10(2),
    "n^2 * 2^n": lambda n: 2 * math.log10(n) + n * math.log10(2),
    "n!":        lambda n: math.log10(math.factorial(n)),
}
sizes = [10, 20, 50, 100, 1000]

# time = operations x 1 µs   ->   log10(time) = log10(operations) + log10(1e-6)
table = pd.DataFrame({f"n = {n}": [log10_time_to_text(f(n) + math.log10(MICROSECOND)) for f in growth.values()] for n in sizes},
                     index=pd.Index(growth.keys(), name="operations"))
print("Running time at 1 µs per operation (age of the Universe = 13.8 billion years):")
table

In [ ]:
# The same as a picture: n from 2 to 100, logarithmic vertical axis
ns = np.arange(2, 101)
plt.figure(figsize=(9, 5.5))
for name, f in growth.items():
    plt.plot(ns, [10 ** (f(n) + math.log10(MICROSECOND)) for n in ns], label=name)
plt.yscale("log")
for label, value in {"1 second": 1, "1 hour": 3600, "1 year": YEAR, "age of the Universe": AGE_OF_UNIVERSE}.items():
    plt.axhline(value, ls=":", color="gray"); plt.text(3, value * 2, label, color="gray")
plt.ylim(1e-7, 1e40)
plt.xlabel("input size n"); plt.ylabel("time [s] at 1 µs per operation"); plt.grid(True, which="both", alpha=0.3)
plt.legend(loc="upper left", ncol=2)
plt.title("Growth classes: polynomial (bottom) vs. exponential and factorial (top)")
plt.show()

Two things to remember from the picture:

* On a logarithmic axis a polynomial bends *down*, an exponential is a straight line, a factorial bends *up*.
* For polynomial algorithms a 1000× faster computer buys a 1000× (n), 32× (n²) or 10× (n³) larger input.
  For $2^n$ it buys **+10 items**, for $n!$ **+2–3 items**. That is the practical meaning of the P vs NP question below.

## 1. Subset sum

**Assignment:** given a list of numbers and a target, is there a subset whose sum equals the target?

*Example:* invoices `[12, 7, 25, 3, 18, 31]`, a bank transfer of `43` – which invoices were paid?
(Accountants really solve this.)

In [ ]:
def verify_subset_sum(numbers: list[int], target: int, certificate: tuple[int, ...]) -> bool:
    """Polynomial – O(n): just add the selected numbers and compare. `certificate` = positions of the chosen numbers."""
    return sum(numbers[i] for i in certificate) == target


def solve_subset_sum(numbers: list[int], target: int):
    """Exponential – tries up to 2^n subsets: first all subsets of size 0, then size 1, size 2, ..."""
    n = len(numbers)
    for r in range(n + 1):                                     # subset size r = 0, 1, ..., n
        for subset in itertools.combinations(range(n), r):     # every choice of r positions
            if sum(numbers[i] for i in subset) == target:      # found one -> done
                return subset
    return None                                                # tried everything -> no solution


invoices = [12, 7, 25, 3, 18, 31]
solution = solve_subset_sum(invoices, 43)
print("Solution:", solution, "->", [invoices[i] for i in solution])
print("Verified:", verify_subset_sum(invoices, 43, solution))

### How does the time grow?

To measure the worst case we use an instance that has **no** solution (all numbers even, target odd),
so the brute force has to try all $2^n$ subsets.

In [ ]:
rng = random.Random(0)
times = {}
for n in range(4, 21):
    numbers = [2 * rng.randint(1, 10 ** 6) for _ in range(n)]   # n random EVEN numbers
    t0 = perf_counter()
    solve_subset_sum(numbers, 1)          # odd target -> impossible -> all 2^n subsets are checked
    times[n] = perf_counter() - t0
    print(f"n = {n:2d}: {2 ** n:>9,} subsets, {times[n]:.4f} s")

n_max = max(times)
time_per_subset = times[n_max] / 2 ** n_max      # time for one subset, from the largest measurement

plt.figure(figsize=(8, 5))
plt.semilogy(list(times), list(times.values()), "s-", label="measured")
ns = np.arange(4, 100)
plt.semilogy(ns, [2.0 ** n * time_per_subset for n in ns], "--", label="2^n extrapolation")
for label, value in {"1 hour": 3600, "1 year": YEAR, "age of the Universe": AGE_OF_UNIVERSE}.items():
    plt.axhline(value, ls=":", color="gray"); plt.text(5, value * 2, label, color="gray")
plt.xlabel("number of items n"); plt.ylabel("time [s]"); plt.grid(True, which="both", alpha=0.3); plt.legend()
plt.title("Subset sum by brute force")
plt.show()

n_universe = next(n for n in ns if 2.0 ** n * time_per_subset > AGE_OF_UNIVERSE)
print(f"Brute force exceeds the age of the Universe at n = {n_universe} items.")
for n in (30, 40, 50, 60, 80, 100):
    print(f"n = {n:3d}: {human_time(2.0 ** n * time_per_subset)}")

In [ ]:
# ...while verification stays instant even for a million numbers
numbers = [rng.randint(1, 10 ** 9) for _ in range(1_000_000)]      # a million random numbers
certificate = tuple(rng.sample(range(len(numbers)), 1000))         # a proposed answer: 1000 of them
target = sum(numbers[i] for i in certificate)                      # (we make the target match, so the answer is correct)

t0 = perf_counter()
ok = verify_subset_sum(numbers, target, certificate)
print(f"n = 1 000 000: verified = {ok} in {1000 * (perf_counter() - t0):.2f} ms")

(Subset sum has a dynamic-programming algorithm with complexity $O(n \cdot \text{target})$.
That looks polynomial, but the *target* is a number, not a size: with 64-bit numbers it is $2^{64}$ steps again.
Such problems are called *weakly* NP-complete – it is the reason the knapsack cryptosystem was broken, and a common exam question.)

## 2. Same graph, two questions: Euler vs. Hamilton

Both questions ask for a closed walk through the graph that visits *everything exactly once*:

* **Euler circuit** – every **edge** exactly once (the seven bridges of Königsberg, snow-plough routes, DNA sequencing).
  Euler proved in 1736: it exists iff the graph is connected and every vertex has an even degree. That is $O(V + E)$ – **P**.
* **Hamiltonian cycle** – every **vertex** exactly once. Nobody knows anything better than (cleverly) trying the permutations – **NP-complete**.
  TSP is its optimisation version: a Hamiltonian cycle with minimum total length.

In [ ]:
def has_euler_circuit(G: nx.Graph) -> bool:
    """Euler's theorem: connected + every vertex has an even number of edges. Just count."""
    return nx.is_connected(G) and all(degree % 2 == 0 for _, degree in G.degree())


def count_hamiltonian_cycles(G: nx.Graph) -> int:
    """Brute force: try every ordering of the vertices and check that consecutive ones are connected."""
    nodes = list(G)
    first, count = nodes[0], 0
    for perm in itertools.permutations(nodes[1:]):          # (n-1)! orderings of the other vertices
        cycle = (first, *perm, first)                       # close the cycle
        if all(G.has_edge(a, b) for a, b in zip(cycle, cycle[1:])):   # every consecutive pair must be an edge
            count += 1
    return count


rows = []
for n in range(5, 11):
    G = nx.random_regular_graph(4, n, seed=1)      # every vertex has degree 4 -> an Euler circuit exists
    t0 = perf_counter(); euler = has_euler_circuit(G); t_euler = perf_counter() - t0
    t0 = perf_counter(); hamilton = count_hamiltonian_cycles(G); t_hamilton = perf_counter() - t0
    rows.append({"vertices": n, "Euler circuit exists": euler, "Euler time [ms]": 1000 * t_euler,
                 "Hamiltonian cycles": hamilton, "Hamilton time [ms]": 1000 * t_hamilton})

# The Euler test on a huge graph: 20 000 vertices, 200 000 edges
big = nx.gnp_random_graph(20_000, 0.001, seed=1)
t0 = perf_counter(); has_euler_circuit(big); t_big = perf_counter() - t0
print(f"Euler check on a graph with {big.number_of_nodes():,} vertices and {big.number_of_edges():,} edges: {1000 * t_big:.0f} ms")
pd.DataFrame(rows).set_index("vertices")

## 3. One word makes the difference: shortest vs. longest path

**Shortest** path from one corner of a grid to the other: Dijkstra, polynomial (previous lab).

**Longest** *simple* path (no vertex twice) between the same corners: NP-hard. The only general method is to enumerate paths,
and the number of corner-to-corner paths in an $m \times m$ grid explodes: 2, 12, 184, 8 512, 1 262 816, 575 780 564, ...

In [ ]:
rows = []
for m in range(2, 6):
    grid = nx.grid_2d_graph(m, m)                    # an m x m grid of vertices
    s, t = (0, 0), (m - 1, m - 1)                    # opposite corners

    t0 = perf_counter()
    shortest = nx.shortest_path_length(grid, s, t)   # breadth-first search: fast
    t_short = perf_counter() - t0

    t0 = perf_counter()
    n_paths, longest = 0, 0
    for path in nx.all_simple_paths(grid, s, t):     # enumerate EVERY path without repeated vertices ...
        n_paths += 1
        longest = max(longest, len(path) - 1)        # ... and remember the longest one
    t_long = perf_counter() - t0

    rows.append({"grid": f"{m}x{m}", "shortest path": shortest, "time [ms]": 1000 * t_short,
                 "longest path": longest, "paths enumerated": n_paths, "time [s]": t_long})

time_per_path = t_long / n_paths
for m, n_paths in [(6, 575_780_564), (7, 789_360_053_252), (8, 3_266_598_486_981_642)]:
    print(f"{m}x{m}: {n_paths:,} paths -> about {human_time(n_paths * time_per_path)}")
pd.DataFrame(rows).set_index("grid")

## 4. Easy and hard siblings – tasks everybody knows

Many hard problems have an easy sibling: the assignment sounds almost the same, one condition changes –
and minutes turn into eternity. You do not need the textbook names to recognise them (they are in brackets for reference):

| everyday task | easy version – P | hard version – NP-complete / NP-hard | what changed |
|---|---|---|---|
| Delivery | the fastest route from the depot to **one** address (shortest path – Dijkstra, A*) | the shortest route that visits **all 20** addresses (TSP) | one destination → all of them |
| Putting things in order | sort the addresses alphabetically (sorting) | put them into the best delivery order (TSP again) | order by a key → order by mutual distances |
| Walking a district | walk along **every street** exactly once – postman, snow plough (Euler circuit) | visit **every crossroads** exactly once (Hamiltonian cycle) | streets → crossroads |
| Bookkeeping | add up all the invoices | find **which** invoices add up to the payment of 43 (subset sum) | sum everything → pick a subset |
| Loading a van | take the most valuable load when goods can be **split** – sand, liquids (fractional knapsack) | the same with **indivisible** boxes (0/1 knapsack) | divisible → indivisible |
| Exam timetable | put exams into **two** slots so that no student has two at once (2-colouring) | the same with **three** slots (3-colouring) | 2 → 3 |
| Forming teams | split people into **pairs** who get along (matching) | split them into **triples** who get along (3-dimensional matching) | 2 → 3 |
| Laying cable | connect the villages as cheaply as possible (minimum spanning tree) | the same, but you may also build **new junction boxes** anywhere (Steiner tree) | extra freedom, harder problem |
| Hiking | the shortest path across the park | the longest walk that never visits a spot twice (longest simple path) | shortest → longest |
| Sudoku | check that a filled-in grid is correct | fill in an empty grid (general n²×n² sudoku) | verify → solve |
| Arithmetic | multiply two 300-digit numbers | split a 600-digit number into its two prime factors (factoring – RSA relies on it; in NP, probably not NP-complete) | forwards → backwards |

The pattern: a tiny change in the wording, an enormous change in difficulty. Some hard versions are small enough in
everyday life to be solved anyway (a 9×9 sudoku, a van with ten boxes) – NP-hardness speaks about the worst case as the size grows.

**Why does it matter in practice?**

* Scheduling (timetables, shifts, CNC jobs), vehicle routing, cutting stock, chip layout, protein folding – all NP-hard.
* You will *not* find an exact fast algorithm. Instead: heuristics, approximation algorithms (Christofides: at most 1.5× optimal),
  local search (2-opt, simulated annealing, genetic algorithms), or solvers (SAT / MIP / constraint programming) that are exponential in the worst case but fast on typical instances.
* Cryptography relies on it the other way round: we *want* factoring and discrete logarithm to be hard.

## Exercise: graph 3-colouring

**Assignment:** colour the vertices of a graph with 3 colours so that no edge connects two vertices of the same colour
(register allocation in compilers, frequency assignment for mobile transmitters, exam timetabling).

1. Implement `verify_colouring` – the polynomial verifier.
2. Implement `solve_3_colouring` by brute force over all $3^n$ colourings (`itertools.product`).
3. Test on the Petersen graph (3-colourable) and on $K_4$ (not 3-colourable). Measure how the time grows with `nx.cycle_graph(n)`.

In [ ]:
def verify_colouring(G: nx.Graph, colouring: dict) -> bool:
    """Returns True if no edge connects two vertices of the same colour. `colouring` = {vertex: colour}."""
    # YOUR CODE HERE
    return False


def solve_3_colouring(G: nx.Graph):
    """Returns a valid colouring {vertex: colour} or None if the graph is not 3-colourable."""
    nodes = list(G)
    # YOUR CODE HERE: for colours in itertools.product(range(3), repeat=len(nodes)): ...
    return None


for name, G in [("Petersen graph", nx.petersen_graph()), ("K4", nx.complete_graph(4))]:
    t0 = perf_counter()
    colouring = solve_3_colouring(G)
    print(f"{name}: {'colourable' if colouring else 'not 3-colourable / not implemented'} "
          f"({perf_counter() - t0:.3f} s)", "verified:" if colouring else "", verify_colouring(G, colouring) if colouring else "")

if (colouring := solve_3_colouring(nx.petersen_graph())):
    nx.draw(nx.petersen_graph(), node_color=[colouring[v] for v in nx.petersen_graph()], cmap="Set1", with_labels=True)